In [11]:
import json
from collections import defaultdict


def extract_entities(text, labels):
    """Extracts named entities from text based on B- and I- tags."""
    tokens = text.split()
    entities = defaultdict(list)

    current_entity = []
    current_tag = None

    for token, label in zip(tokens, labels):
        if label == "O":
            if current_entity and current_tag:
                entities[current_tag].append(" ".join(current_entity))
                current_entity = []
                current_tag = None
            continue

        # Split label into prefix (B/I) and entity type (e.g., Symptom, Medicine)
        prefix, entity_type = label.split("-", 1)

        if prefix == "B":
            # Save the previous entity if it exists
            if current_entity and current_tag:
                entities[current_tag].append(" ".join(current_entity))

            # Start new entity
            current_entity = [token]
            current_tag = entity_type

        elif prefix == "I":
            # Continue current entity if tag matches, otherwise handle edge case
            if current_tag == entity_type:
                current_entity.append(token)
            else:
                if current_entity and current_tag:
                    entities[current_tag].append(" ".join(current_entity))
                current_entity = [token]
                current_tag = entity_type

    # Catch any remaining entity at the end of the sequence
    if current_entity and current_tag:
        entities[current_tag].append(" ".join(current_entity))

    # Deduplicate extracted entities per record while preserving order
    return {
        tag: list(dict.fromkeys(values)) for tag, values in entities.items()
    }


def transform_dataset(input_file, output_file):
    with open(input_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    # First pass: discover all unique tag types across the entire dataset
    all_tag_types = set()
    for item in data:
        for label in item.get("labels", []):
            if label != "O" and "-" in label:
                _, tag_type = label.split("-", 1)
                all_tag_types.add(tag_type)

    transformed_data = []

    # Second pass: construct new JSON objects with explicit columns for each tag
    for item in data:
        text = item.get("text", "")
        labels = item.get("labels", [])

        extracted = extract_entities(text, labels)

        # Base record with original text
        new_record = {"text": text}

        # Populate columns for ALL discovered tags (empty list if entity isn't present)
        for tag_type in sorted(all_tag_types):
            new_record[tag_type] = extracted.get(tag_type, [])

        transformed_data.append(new_record)

    # Save transformed JSON
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(transformed_data, f, ensure_ascii=False, indent=2)

    print(
        f"Successfully transformed {len(data)} records across tag columns: {list(all_tag_types)}"
    )


# Run script (replace with your file paths)
transform_dataset("train.json", "transformed_data.json")

Successfully transformed 25426 records across tag columns: ['Health Condition', 'Symptom', 'Dosage', 'Specialist', 'Age', 'Medicine', 'Medical Procedure']
